# ML models per treatment arm: RF and XGBoost on the continuous outcome

The same pipeline runs in all three arm notebooks (`ml_mexiletine`, `ml_oligonucleotide`, `ml_tideglusib`). Only the **Parameters** cell differs. All logic lives in `ml_pipeline.py`, and arm settings (outcome, covariate list) live in `arm_config.py`.

1. **Data.** Load the shared train/test split and keep this arm. A leakage guard rejects IDs, outcomes and responder labels as inputs.
2. **Nested cross-validation (training set only).** Outer: 5-fold × 3 repeats, stratified on outcome quartiles; it scores each model. Inner: 5-fold, also stratified; it tunes hyperparameters. Models: null (training mean), RF untuned/tuned, XGBoost untuned/tuned.
3. **Model selection.** The model with the lowest mean outer-fold RMSE, plus paired fold-by-fold comparisons.
4. **Final models.** Re-tune on the full training set, then score **once** on the locked test set (bootstrap 95% CIs).
5. **SHAP covariate ranking** for the selected model: mean |SHAP| on the training set, its stability across the outer CV folds, beeswarm and dependence plots.
6. **Save** tables and figures to `outputs/<arm>/`.

In [ ]:
# ---- Parameters: the only cell that differs between the three arm notebooks ----
ARM = "Mexiletine"
N_NOISE_PROBES = 0   # > 0 adds pure-noise columns as a reference level for covariate importance
QUICK_RUN = False    # True = fewer folds and bootstraps, only for checking the code runs

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import arm_config as cfg
import ml_pipeline as mlp

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ARM_CFG = cfg.ARMS[ARM]
CV = dict(cfg.CV_SETTINGS)
N_BOOT = 2000
OUT_DIR = cfg.OUTPUT_ROOT / ARM_CFG["slug"]
if QUICK_RUN:
    CV.update(outer_splits=3, outer_repeats=1, inner_splits=3, final_splits=3, final_repeats=1)
    N_BOOT = 200
    OUT_DIR = cfg.OUTPUT_ROOT / f"{ARM_CFG['slug']}_quick"

print(f"{ARM}: outcome = {ARM_CFG['outcome']} ({ARM_CFG['better']} is better)")
print("Split files:", cfg.TRAIN_PATH.name, "/", cfg.TEST_PATH.name)
print("Outputs ->", OUT_DIR)

## 1. Data

In [ ]:
data = mlp.load_arm_data(
    cfg.TRAIN_PATH, cfg.TEST_PATH, ARM, ARM_CFG["outcome"], ARM_CFG["covariates"],
    cat_cols_all=cfg.CAT_COLS, leaky_cols=cfg.LEAKY_COLS, arm_col=cfg.ARM_COL, id_col=cfg.ID_COL,
)
data = mlp.add_noise_probes(data, N_NOISE_PROBES, cfg.SEED)

print(f"{len(data.covariates)} model inputs: {data.covariates}")
print("Dropped:", data.dropped or "none")
mlp.data_summary(data)

## 2. Models and nested cross-validation

All models are scored on the **same outer folds**, so their scores can be compared fold by fold. Tuning happens only on the outer-training portion of each fold.

In [ ]:
specs = mlp.default_model_specs(cfg.SEED)
MODEL_ORDER = list(specs)
pd.DataFrame(
    {name: {"family": s.family, "grid": s.grid or "fixed settings"} for name, s in specs.items()}
).T

In [ ]:
cv_res = mlp.nested_cv(
    data, specs, outer_splits=CV["outer_splits"], outer_repeats=CV["outer_repeats"],
    inner_splits=CV["inner_splits"], n_bins=CV["n_bins"], seed=cfg.SEED,
)
mlp.summarize_cv(cv_res.scores, MODEL_ORDER)

In [ ]:
fig_cv = mlp.plot_cv_scores(cv_res.scores, MODEL_ORDER, title=f"{ARM}: nested CV (training set)")

## 3. Model selection

The model with the lowest mean outer-fold RMSE is selected. The paired table shows how much worse each other model is on the same folds, and in how many folds the selected model wins. With about 27 patients per validation fold, small differences are not meaningful.

In [ ]:
best, paired = mlp.select_best(cv_res.scores, metric="RMSE")
print(f"Selected model (lowest mean outer-fold RMSE): {best}")
paired

In [ ]:
# How consistently tuning picked the same settings across outer folds
mlp.tuned_param_counts(cv_res.params)

## 4. Final models and the locked test set

Each model is re-tuned on the full training set, then scored once on the test set. Selection was made on CV above, not on these test numbers.

In [ ]:
final_models, final_params = mlp.fit_final(
    data, specs, final_splits=CV["final_splits"], final_repeats=CV["final_repeats"],
    n_bins=CV["n_bins"], seed=cfg.SEED,
)
final_params

In [ ]:
test_metrics, test_preds = mlp.evaluate_test(final_models, data, n_boot=N_BOOT, seed=cfg.SEED)
test_metrics.reindex(MODEL_ORDER)

In [ ]:
plot_models = list(dict.fromkeys(["rf_tuned", "xgb_tuned", best]))
fig_pred = mlp.plot_pred_vs_obs(
    test_preds, plot_models, ARM_CFG["outcome_label"], title=f"{ARM}: test set (n = {len(data.y_test)})"
)

## 5. SHAP covariate ranking (selected model)

SHAP values for one-hot-encoded categoricals are summed back to the original covariate. The ranking uses the final model on the training set. `cv_median_rank` and `cv_top3_freq` show how stable each covariate's rank is across the 15 outer CV folds. SHAP explains the model, not the biology.

In [ ]:
shap_train = mlp.shap_explain(final_models[best], data.X_train, data)
shap_test = mlp.shap_explain(final_models[best], data.X_test, data)
ranking = mlp.shap_ranking(shap_train, data, shap_test, cv_res.fold_shap, best)
ranking

In [ ]:
fig_imp = mlp.plot_shap_importance(ranking, cv_res.fold_shap, best, title=f"{ARM}: SHAP importance ({best})")

In [ ]:
fig_bee = mlp.plot_shap_beeswarm(shap_train, title=f"{ARM}: SHAP values ({best}, training set)")

In [ ]:
top_covs = [c for c in ranking.index if c not in data.noise_cols][:3]
fig_dep = mlp.plot_shap_dependence(shap_train, data, top_covs, title=f"{ARM}: SHAP dependence, top covariates")

## 6. Save outputs

In [ ]:
tables = {  # name: (DataFrame, write index?)
    "cv_scores": (cv_res.scores, False),
    "cv_summary": (mlp.summarize_cv(cv_res.scores, MODEL_ORDER), True),
    "cv_tuned_params": (cv_res.params, False),
    "cv_paired_vs_best": (paired, True),
    "cv_fold_shap": (cv_res.fold_shap, False),
    "final_tuned_params": (final_params, False),
    "test_metrics": (test_metrics, True),
    "test_predictions": (test_preds, False),
    "shap_ranking": (ranking, True),
    "shap_values_train": (shap_train.by_covariate.assign(patient_id=data.ids_train.to_numpy()), False),
}
for name, (df, write_index) in tables.items():
    mlp.save_table(df, OUT_DIR, name, index=write_index)

figures = {
    "cv_scores": fig_cv, "test_pred_vs_obs": fig_pred, "shap_importance": fig_imp,
    "shap_beeswarm": fig_bee, "shap_dependence": fig_dep,
}
for name, fig in figures.items():
    mlp.save_fig(fig, OUT_DIR, name)

mlp.save_run_info(
    OUT_DIR, arm=ARM, outcome=ARM_CFG["outcome"], better=ARM_CFG["better"], selected_model=best,
    seed=cfg.SEED, cv=CV, n_boot=N_BOOT, quick_run=QUICK_RUN, covariates=data.covariates,
    dropped=data.dropped, noise_probes=data.noise_cols,
    grids={name: s.grid for name, s in specs.items()},
    train_file=str(cfg.TRAIN_PATH), test_file=str(cfg.TEST_PATH),
)
print(f"Saved {len(tables)} tables and {len(figures)} figures to {OUT_DIR}")

## Notes (fill in after running)

- Selected model and how clearly it beats the alternatives (paired folds):
- Test-set performance and its uncertainty:
- Top covariates, their direction, and how stable their rank is across folds:
- Clinical interpretation: